# MDU — Fəsil 19: Dərin öyrənmə əsaslı kompüter görməsi

Bu notebook real görüntü üzərində YOLO inference, CNN transfer learning skeleti və detection metriklərinin hesablanmasını nümayiş etdirir.

## 1. Mühit və GPU yoxlaması

In [ ]:
!pip -q install ultralytics torchvision matplotlib opencv-python

In [ ]:
import torch, cv2, numpy as np, matplotlib.pyplot as plt
print('PyTorch:', torch.__version__)
print('OpenCV:', cv2.__version__)
print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available(): print(torch.cuda.get_device_name(0))

## 2. Real demo şəkli ilə YOLO inference
Ultralytics-in ictimai demo avtobus təsviri istifadə olunur. Model çəkiləri ilk çağırışda endirilə bilər.

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image
import requests
url='https://ultralytics.com/images/bus.jpg'
r=requests.get(url,timeout=30); r.raise_for_status()
open('/content/bus.jpg','wb').write(r.content)
model=YOLO('yolo26n.pt')
result=model.predict('/content/bus.jpg',conf=0.25)[0]
annotated=result.plot()
display(Image.fromarray(annotated[:,:,::-1]))
print(result.boxes)

## 3. Confidence threshold müqayisəsi

In [ ]:
for conf in [0.15,0.25,0.50,0.70]:
    r=model.predict('/content/bus.jpg',conf=conf,verbose=False)[0]
    print(f'conf={conf:.2f} -> detection sayı={len(r.boxes)}')

## 4. CNN transfer learning — ImageNet pretrained ResNet18
Bu nümunə classification üçündür. Detection üçün bounding-box annotasiyası və detection arxitekturası ayrıca tələb olunur.

In [ ]:
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
weights=ResNet18_Weights.DEFAULT
classifier=resnet18(weights=weights)
for p in classifier.parameters(): p.requires_grad=False
num_classes=3
classifier.fc=nn.Linear(classifier.fc.in_features,num_classes)
print(classifier.fc)
print('Trainable parameters:',sum(p.numel() for p in classifier.parameters() if p.requires_grad))

## 5. IoU, Precision, Recall və F1 hesablanması

In [ ]:
def box_iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1])
    x2=min(a[2],b[2]); y2=min(a[3],b[3])
    inter=max(0,x2-x1)*max(0,y2-y1)
    area_a=max(0,a[2]-a[0])*max(0,a[3]-a[1])
    area_b=max(0,b[2]-b[0])*max(0,b[3]-b[1])
    union=area_a+area_b-inter
    return inter/union if union else 0

gt=[30,30,150,130]
pred=[70,50,180,150]
print('IoU:',round(box_iou(gt,pred),4))
TP,FP,FN=72,8,18
precision=TP/(TP+FP) if TP+FP else 0
recall=TP/(TP+FN) if TP+FN else 0
f1=2*precision*recall/(precision+recall) if precision+recall else 0
print('Precision:',precision,'Recall:',recall,'F1:',f1)

## 6. Öz datasetinizlə YOLO təlimi
`data.yaml` faylında train/val yolları və sinif adları olmalıdır. Aşağıdakı kodu yalnız annotasiya olunmuş datasetinizi Colab-a yüklədikdən sonra icra edin.

In [ ]:
# Öz datasetiniz üçün nümunə — əvvəlcə /content/dataset/data.yaml hazırlayın
# custom = YOLO('yolo26n.pt')
# custom.train(data='/content/dataset/data.yaml', epochs=30, imgsz=640, batch=8)
# metrics = custom.val(data='/content/dataset/data.yaml')
# print('mAP50:', metrics.box.map50, 'mAP50-95:', metrics.box.map)

## 7. Laboratoriya hesabatı
1. Datasetin mənbəyini, lisenziyasını və siniflərini göstərin.
2. Train/validation/test bölgüsünü izah edin; eyni video kadrlarını müxtəlif bölmələrə qarışdırmayın.
3. Confidence threshold dəyişdikdə detection sayının necə dəyişdiyini qeyd edin.
4. IoU, Precision, Recall, F1 və mAP nəticələrini izah edin.
5. Ən azı 3 FP və ya FN nümunəsini təhlil edin.
6. Modelin real mühitdəki məhdudiyyətlərini yazın.